In [ ]:
# Standard library imports
import os
import sys

# Third-party imports
import numpy as np
import pandas as pd

# Local imports
module_path = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from paths import BASE_INPUT_PATH, BASE_OUTPUT_PATH

In [ ]:
def errors_calculation(dataset, model, product, product_sign, price_type, output_folder, lookback=None, 
                      lstm_strategy=None):
    """
    Calculate prediction errors and metrics for model results.
    
    Parameters:
    -----------
    lstm_strategy : str
        For LSTM models, which prediction to use: 'ensemble', 'recency', or 'single'
    lookback : int
        Lookback period used for training (42 or 168 for LSTM models)
    """
    # Look for files containing all the specified parameters
    # Define more specific model folder paths
    model_folder_path = os.path.join(output_folder, f"{model}_model_output")
    
    # Check if the specific model folder exists
    if os.path.exists(model_folder_path):
        # Look only in the specific model folder
        search_path = model_folder_path
    else:
        # Fallback to searching in all folders
        search_path = output_folder
    
    # Get matching files from the search path
    # Include lookback in pattern for LSTM models
    if model in ['lstm', 'bidirectional_lstm', 'lstm_feature_1', 'lstm_feature', 'lstm_features_1_3', 'lstm_features_1_4'] and lookback is not None:
        pattern = f"{model}_ts{lookback}_model_results_{product}_{product_sign}_{price_type}"
    else:
        pattern = f"{model}_model_results_{product}_{product_sign}_{price_type}"
    
    matching_files = []
    # Walk through the directory structure
    for root, dirs, files in os.walk(search_path):
        for file in files:
            full_path = os.path.join(root, file)
            if pattern in file and file.endswith('.csv'):
                matching_files.append(full_path)
    
    if len(matching_files) == 0:
        lookback_msg = f", lookback={lookback}" if lookback is not None else ""
        print(f"No matching files found for {model}, {product}, {product_sign}, {price_type}{lookback_msg}")
        return None
    elif len(matching_files) > 1:
        # Take the most recent file (based on file modification time)
        matching_files.sort(key=os.path.getmtime, reverse=True)
        results_file = matching_files[0]
        print(f"Multiple files found for {model}, {product}, {product_sign}, {price_type}. Using the most recent one: {os.path.basename(results_file)}")
    else:
        results_file = matching_files[0]
    
    # Read the results file
    try:
        results = pd.read_csv(results_file)
        # Check if 'DATE' column exists
        if 'DATE' in results.columns:
            results['DATE'] = pd.to_datetime(results['DATE'])
            results.set_index('DATE', inplace=True)
        # For files without a DATE column (like 00_04 files which may use a different format)
        elif 'date' in results.columns:
            results['date'] = pd.to_datetime(results['date'])
            results.set_index('date', inplace=True)
        
        # Filter data from beginning of July 2024 onwards
        results = results[results.index >= '2024-07-01']
        
    except Exception as e:
        print(f"Error reading file {results_file}: {e}")
        return None

    # Handle different prediction column formats based on model type
    if model in ['lstm', 'bidirectional_lstm', 'lstm_feature_1', 'lstm_feature', 'lstm_features_1_3', 'lstm_features_1_4']:
        # LSTM models have different prediction columns - select based on strategy
        if lstm_strategy == 'ensemble' and 'D+1_ENSEMBLE_MODEL' in results.columns:
            results['D+1'] = results['D+1_ENSEMBLE_MODEL']
            print(f"Using D+1_ENSEMBLE_MODEL for {model}, {product}, {product_sign}")
        elif lstm_strategy == 'recency' and 'D+1_RECENCY_WEIGHTED' in results.columns:
            results['D+1'] = results['D+1_RECENCY_WEIGHTED']
            print(f"Using D+1_RECENCY_WEIGHTED for {model}, {product}, {product_sign}")
        elif lstm_strategy == 'single' and 'D+1_SINGLE_MODEL' in results.columns:
            results['D+1'] = results['D+1_SINGLE_MODEL']
            print(f"Using D+1_SINGLE_MODEL for {model}, {product}, {product_sign}")
        else:
            # Try to find any D+1 column
            d1_columns = [col for col in results.columns if 'D+1' in col]
            if d1_columns:
                print(f"No matching column for strategy {lstm_strategy}. Using {d1_columns[0]} for {model} model")
                results['D+1'] = results[d1_columns[0]]
            else:
                print(f"No D+1 prediction column found in {results_file}")
                return None
    elif 'D+1' not in results.columns:
        # For other models, look for alternative column names
        pred_column = None
        for col in results.columns:
            if col.lower() in ['d+1', 'pred', 'prediction', 'forecast']:
                pred_column = col
                break
        
        if pred_column is None:
            print(f"No prediction column found in {results_file}")
            return None
        
        # Rename to standardized column
        results['D+1'] = results[pred_column]

    # Filter the dataset for the specified product
    dataset = dataset.loc[dataset['PRODUCT'] == product]

    # Ensure matching dates between results and dataset
    marginal_column = f'{product_sign}_GERMANY_MARGINAL_CAPACITY_PRICE_[(EUR/MW)/h]'
    y = dataset[marginal_column]
    marginal_df = pd.DataFrame({marginal_column: y})
    marginal_df = marginal_df.loc[marginal_df.index.isin(results.index)]

    # Remove D+1_* columns except D+1 which we want to keep
    for col in results.columns:
        if col.startswith('D+1_') and col not in ['D+1']:
            results = results.drop(columns=col)
    
    results['MARGINAL_PRICE'] = marginal_df[marginal_column].values
    results['RMSE'] = (results['D+1'] - results['ACTUAL_VALUE'])**2
    results['MAPE'] = np.abs((results['D+1'] - results['ACTUAL_VALUE']) / results['ACTUAL_VALUE'])
    results['MAE'] = np.abs(results['D+1'] - results['ACTUAL_VALUE'])
    results['ACCEPTANCE'] = np.where(results['D+1'] <= results['MARGINAL_PRICE'], 1, 0)
    results['REVENUE'] = np.where(results['ACCEPTANCE'] == 1, results['D+1'], 0)

    if model in ['lstm', 'bidirectional_lstm', 'lstm_feature_1', 'lstm_feature', 'lstm_features_1_3', 'lstm_features_1_4'] and lookback is not None:
        pattern = f"{model}_ts{lookback}_{lstm_strategy}_model_processed_results_{product}_{product_sign}_{price_type}"
    else:
        pattern = f"{model}_model_processed_results_{product}_{product_sign}_{price_type}"
    
    # Ensure the output directory exists
    processed_results_dir = os.path.join(model_folder_path, 'processed_results')
    os.makedirs(processed_results_dir, exist_ok=True)
    
    # Save the processed results
    results.to_csv(os.path.join(processed_results_dir, f'{pattern}.csv'), index=True)

    # Create model name with additional info if applicable
    model_name = model
    if lookback is not None:
        model_name += f"_ts{lookback}"
    if lstm_strategy is not None:
        model_name += f"_{lstm_strategy}"
    
    # Create a dataframe from dictionary
    new_row = {
        'MODEL': model_name,
        'PRODUCT': product,
        'PRODUCT_SIGN': product_sign,
        'PRICE_TYPE': price_type,
        'RMSE': np.sqrt(results['RMSE'].mean()),
        'MAPE': results['MAPE'].mean(),
        'MAE': results['MAE'].mean(),
        'MED_MAE': results['MAE'].median(),
        'ZERO_COUNT': (results['D+1'].isnull() | (results['D+1'] == 0)).sum(),
        'NEGATIVE_COUNT': (results['D+1'] < 0).sum(),
        'ACCEPTANCE': results['ACCEPTANCE'].sum(),
        'CHANCE_OF_ACCEPTANCE': results['ACCEPTANCE'].mean(),
        'REVENUE': results['REVENUE'].sum(),
        'MAX_REVENUE': results['ACTUAL_VALUE'].sum(),
        'ACHIEVED_REVENUE_PERCENTAGE': (results['REVENUE'].sum() / results['ACTUAL_VALUE'].sum()) * 100 if results['ACTUAL_VALUE'].sum() != 0 else np.nan
    }
    
    # Return as a single-row DataFrame
    return pd.DataFrame([new_row])

In [ ]:
# Create output folder if it doesn't exist
output_folder = BASE_OUTPUT_PATH / 'postprocessed_data'
os.makedirs(output_folder, exist_ok=True)

# Main execution code
all_models_comparison = pd.DataFrame()

models = ['arma', 'sarma',
          'armax_exog_1', 'armax', 'armax_exog_1_3', 'armax_exog_1_4',
          'sarmax_exog_1', 'sarmax', 'sarmax_exog_1_3', 'sarmax_exog_1_4',
          'rw_arma_3m', 'rw_arma_6m', 'rw_arma_12m',
          'rw_armax_exog_1_3m', 'rw_armax_exog_1_6m', 'rw_armax_exog_1_12m',
          'rw_armax_3m', 'rw_armax_6m', 'rw_armax_12m',
          'rw_armax_exog_1_3_3m', 'rw_armax_exog_1_3_6m', 'rw_armax_exog_1_3_12m',
          'rw_armax_exog_1_4_3m', 'rw_armax_exog_1_4_6m', 'rw_armax_exog_1_4_12m',
          'rw_sarma_3m', 'rw_sarma_6m', 'rw_sarma_12m',
          'rw_sarmax_exog_1_3m', 'rw_sarmax_exog_1_6m', 'rw_sarmax_exog_1_12m',
        #   'rw_sarmax_3m', 'rw_sarmax_6m', 'rw_sarmax_12m',
        #   'rw_sarmax_exog_1_3_3m', 'rw_sarmax_exog_1_3_6m', 'rw_sarmax_exog_1_3_12m',
        #   'rw_sarmax_exog_1_4_3m', 'rw_sarmax_exog_1_4_6m', 'rw_sarmax_exog_1_4_12m',
          'lstm', 'bidirectional_lstm',
          'lstm_feature_1', 'lstm_feature', 'lstm_features_1_3', 'lstm_features_1_4'
          ]
products = ['00_04', '04_08', '08_12', '12_16', '16_20', '20_24']
product_signs = ['NEG', 'POS']
price_types = ['AVERAGE', 'MARGINAL']
lstm_strategies = ['ensemble', 'recency', 'single']
lookbacks = [42, 168]  # Lookback periods for LSTM models

# Check if dataset exists before reading
dataset_path = BASE_INPUT_PATH / 'processed_afrr_data.csv'
if not dataset_path.exists():
    print(f"Dataset not found at {dataset_path}")
    exit(1)

# Read the dataset
try:
    processed_afrr_data = pd.read_csv(dataset_path, parse_dates=['DATE'], index_col=['DATE'])
    # Filter data from beginning of July 2024 onwards
    processed_afrr_data = processed_afrr_data[processed_afrr_data.index >= '2024-07-01']
except Exception as e:
    print(f"Error reading dataset: {e}")
    exit(1)

# Process each model type
for model in models:
    if model in ['lstm', 'bidirectional_lstm', 'lstm_feature_1', 'lstm_feature', 'lstm_features_1_3', 'lstm_features_1_4']:
        # For LSTM models, process all combinations of lookback and strategy
        for lookback in lookbacks:
            for lstm_strategy in lstm_strategies:
                for product in products:
                    for product_sign in product_signs:
                        for price_type in price_types:
                            print(f"Processing {model} with ts{lookback}, {lstm_strategy} strategy, {product}, {product_sign}, {price_type}")
                            
                            postprocessed_df = errors_calculation(
                                dataset=processed_afrr_data,
                                model=model,
                                product=product,
                                product_sign=product_sign,
                                price_type=price_type,
                                output_folder=BASE_OUTPUT_PATH,
                                lookback=lookback,
                                lstm_strategy=lstm_strategy
                            )
                            
                            if postprocessed_df is not None:
                                all_models_comparison = pd.concat([all_models_comparison, postprocessed_df], ignore_index=True)

    else:
        # For non-LSTM models, just process without lookback and strategy
        for product in products:
            for product_sign in product_signs:
                for price_type in price_types:
                    print(f"Processing {model}, {product}, {product_sign}, {price_type}")
                    
                    postprocessed_df = errors_calculation(
                        dataset=processed_afrr_data,
                        model=model,
                        product=product,
                        product_sign=product_sign,
                        price_type=price_type,
                        output_folder=BASE_OUTPUT_PATH
                    )
                    
                    if postprocessed_df is not None:
                        all_models_comparison = pd.concat([all_models_comparison, postprocessed_df], ignore_index=True)

print(f"Analysis complete.")

In [ ]:
# Min-max normalization with outlier removal (using percentiles)
def robust_normalize(series, upper_percentile=95):
    """Normalize values with outlier handling"""
    cap_value = series.quantile(upper_percentile/100)
    capped_series = series.clip(upper=cap_value)
    return 1 - (capped_series / cap_value)

# Apply to your dataframe
if not all_models_comparison.empty:
    # Using robust normalization with percentile capping
    all_models_comparison['MAPE_NORM'] = robust_normalize(all_models_comparison['MAPE'], 99)
    all_models_comparison['RMSE_NORM'] = robust_normalize(all_models_comparison['RMSE'], 99)
    all_models_comparison['MAE_NORM'] = robust_normalize(all_models_comparison['MAE'], 99)
    
    # Create combined score with equal weights
    all_models_comparison['COMBINED_SCORE'] = (
        0.34 * all_models_comparison['MAPE_NORM'] + 
        0.33 * all_models_comparison['RMSE_NORM'] + 
        0.33 * all_models_comparison['MAE_NORM']
    )

    all_models_comparison['REVENUE'] = all_models_comparison['REVENUE']*0.004
    all_models_comparison['MAX_REVENUE'] = all_models_comparison['MAX_REVENUE']*0.004

In [ ]:
new_names = {
    'rw_arma_3m': 'ARMA (3 Months)',
    'rw_arma_6m': 'ARMA (6 Months)',
    'rw_arma_12m': 'ARMA (12 Months)',
    'rw_armax_exog_1_3m': 'ARMAX (Exog. 1) (3 Months)',
    'rw_armax_exog_1_6m': 'ARMAX (Exog. 1) (6 Months)',
    'rw_armax_exog_1_12m': 'ARMAX (Exog. 1) (12 Months)',
    'rw_armax_3m': 'ARMAX (Exog. 4) (3 Months)',
    'rw_armax_6m': 'ARMAX (Exog. 4) (6 Months)',
    'rw_armax_12m': 'ARMAX (Exog. 4) (12 Months)',
    'rw_armax_exog_1_3_3m': 'ARMAX (Exog. 1&3) (3 Months)',
    'rw_armax_exog_1_3_6m': 'ARMAX (Exog. 1&3) (6 Months)',
    'rw_armax_exog_1_3_12m': 'ARMAX (Exog. 1&3) (12 Months)',
    'rw_armax_exog_1_4_3m': 'ARMAX (Exog. 1&4) (3 Months)',
    'rw_armax_exog_1_4_6m': 'ARMAX (Exog. 1&4) (6 Months)',
    'rw_armax_exog_1_4_12m': 'ARMAX (Exog. 1&4) (12 Months)',
    'rw_sarma_3m': 'SARMA (3 Months)',
    'rw_sarma_6m': 'SARMA (6 Months)',
    'rw_sarma_12m': 'SARMA (12 Months)',
    'rw_sarmax_exog_1_3m': 'SARMAX (Exog. 1) (3 Months)',
    'rw_sarmax_exog_1_6m': 'SARMAX (Exog. 1) (6 Months)',
    'rw_sarmax_exog_1_12m': 'SARMAX (Exog. 1) (12 Months)',
    # 'rw_sarmax_3m': 'SARMAX (Exog. 4) (3 Months)',
    # 'rw_sarmax_6m': 'SARMAX (Exog. 4) (6 Months)',
    # 'rw_sarmax_12m': 'SARMAX (Exog. 4) (12 Months)',
    # 'rw_sarmax_exog_1_3_3m': 'SARMAX (Exog. 1&3) (3 Months)',
    # 'rw_sarmax_exog_1_3_6m': 'SARMAX (Exog. 1&3) (6 Months)',
    # 'rw_sarmax_exog_1_3_12m': 'SARMAX (Exog. 1&3) (12 Months)',
    # 'rw_sarmax_exog_1_4_3m': 'SARMAX (Exog. 1&4) (3 Months)',
    # 'rw_sarmax_exog_1_4_6m': 'SARMAX (Exog. 1&4) (6 Months)',
    # 'rw_sarmax_exog_1_4_12m': 'SARMAX (Exog. 1&4) (12 Months)',
    'arma': 'ARMA',
    'armax_exog_1': 'ARMAX (Exog. 1)',
    'armax': 'ARMAX (Exog. 4)',
    'armax_exog_1_3': 'ARMAX (Exog. 1&3)',
    'armax_exog_1_4': 'ARMAX (Exog. 1&4)',
    'sarma': 'SARMA',
    'sarmax_exog_1': 'SARMAX (Exog. 1)',
    'sarmax': 'SARMAX (Exog. 4)',
    'sarmax_exog_1_3': 'SARMAX (Exog. 1&3)',
    'sarmax_exog_1_4': 'SARMAX (Exog. 1&4)',
    'lstm_ts42_single': 'LSTM (42 TimeSteps, Single)',
    'lstm_ts42_ensemble': 'LSTM (42 TimeSteps, Ensemble)',
    'lstm_ts42_recency': 'LSTM (42 TimeSteps, Weighted Ensemble)',
    'lstm_ts168_single': 'LSTM (168 TimeSteps, Single)',
    'lstm_ts168_ensemble': 'LSTM (168 TimeSteps, Ensemble)',
    'lstm_ts168_recency': 'LSTM (168 TimeSteps, Weighted Ensemble)',
    'lstm_feature_1_ts42_single': 'LSTM with Feature 1 (42 TimeSteps, Single)',
    'lstm_feature_1_ts42_ensemble': 'LSTM with Feature 1 (42 TimeSteps, Ensemble)',
    'lstm_feature_1_ts42_recency': 'LSTM with Feature 1 (42 TimeSteps, Weighted Ensemble)',
    'lstm_feature_1_ts168_single': 'LSTM with Feature 1 (168 TimeSteps, Single)',
    'lstm_feature_1_ts168_ensemble': 'LSTM with Feature 1 (168 TimeSteps, Ensemble)',
    'lstm_feature_1_ts168_recency': 'LSTM with Feature 1 (168 TimeSteps, Weighted Ensemble)',
    'lstm_feature_ts42_single': 'LSTM with Feature 4 (42 TimeSteps, Single)',
    'lstm_feature_ts42_ensemble': 'LSTM with Feature 4 (42 TimeSteps, Ensemble)',
    'lstm_feature_ts42_recency': 'LSTM with Feature 4 (42 TimeSteps, Weighted Ensemble)',
    'lstm_feature_ts168_single': 'LSTM with Feature 4 (168 TimeSteps, Single)',
    'lstm_feature_ts168_ensemble': 'LSTM with Feature 4 (168 TimeSteps, Ensemble)',
    'lstm_feature_ts168_recency': 'LSTM with Feature 4 (168 TimeSteps, Weighted Ensemble)',
    'lstm_features_1_3_ts42_single': 'LSTM with Features 1&3 (42 TimeSteps, Single)',
    'lstm_features_1_3_ts42_ensemble': 'LSTM with Features 1&3 (42 TimeSteps, Ensemble)',
    'lstm_features_1_3_ts42_recency': 'LSTM with Features 1&3 (42 TimeSteps, Weighted Ensemble)',
    'lstm_features_1_3_ts168_single': 'LSTM with Features 1&3 (168 TimeSteps, Single)',
    'lstm_features_1_3_ts168_ensemble': 'LSTM with Features 1&3 (168 TimeSteps, Ensemble)',
    'lstm_features_1_3_ts168_recency': 'LSTM with Features 1&3 (168 TimeSteps, Weighted Ensemble)',
    'lstm_features_1_4_ts42_single': 'LSTM with Features 1&4 (42 TimeSteps, Single)',
    'lstm_features_1_4_ts42_ensemble': 'LSTM with Features 1&4 (42 TimeSteps, Ensemble)',
    'lstm_features_1_4_ts42_recency': 'LSTM with Features 1&4 (42 TimeSteps, Weighted Ensemble)',
    'lstm_features_1_4_ts168_single': 'LSTM with Features 1&4 (168 TimeSteps, Single)',
    'lstm_features_1_4_ts168_ensemble': 'LSTM with Features 1&4 (168 TimeSteps, Ensemble)',
    'lstm_features_1_4_ts168_recency': 'LSTM with Features 1&4 (168 TimeSteps, Weighted Ensemble)',
    'bidirectional_lstm_ts42_single': 'BiLSTM (42 TimeSteps, Single)',
    'bidirectional_lstm_ts42_ensemble': 'BiLSTM (42 TimeSteps, Ensemble)',
    'bidirectional_lstm_ts42_recency': 'BiLSTM (42 TimeSteps, Weighted Ensemble)',
    'bidirectional_lstm_ts168_single': 'BiLSTM (168 TimeSteps, Single)',
    'bidirectional_lstm_ts168_ensemble': 'BiLSTM (168 TimeSteps, Ensemble)',
    'bidirectional_lstm_ts168_recency': 'BiLSTM (168 TimeSteps, Weighted Ensemble)',
}

all_models_comparison['MODEL_NAME'] = all_models_comparison['MODEL'].replace(new_names)

In [ ]:
model_groups = {
    'ROLLING_WINDOW_STATISTICAL': ['rw_arma_3m', 'rw_arma_6m', 'rw_arma_12m',
                                   'rw_armax_exog_1_3m', 'rw_armax_exog_1_6m', 'rw_armax_exog_1_12m',
                                   'rw_armax_3m', 'rw_armax_6m', 'rw_armax_12m',
                                   'rw_armax_exog_1_3_3m', 'rw_armax_exog_1_3_6m', 'rw_armax_exog_1_3_12m',
                                   'rw_armax_exog_1_4_3m', 'rw_armax_exog_1_4_6m', 'rw_armax_exog_1_4_12m',
                                   'rw_sarma_3m', 'rw_sarma_6m', 'rw_sarma_12m',
                                   'rw_sarmax_exog_1_3m', 'rw_sarmax_exog_1_6m', 'rw_sarmax_exog_1_12m',
                                #    'rw_sarmax_3m', 'rw_sarmax_6m', 'rw_sarmax_12m',
                                #   'rw_sarmax_exog_1_3_3m', 'rw_sarmax_exog_1_3_6m', 'rw_sarmax_exog_1_3_12m',
                                #   'rw_sarmax_exog_1_4_3m', 'rw_sarmax_exog_1_4_6m', 'rw_sarmax_exog_1_4_12m'
                                   ],
    'EXPANDING_WINDOW_STATISTICAL': ['arma', 'sarma', 
                                     'armax_exog_1', 'armax', 'armax_exog_1_3', 'armax_exog_1_4',
                                     'sarmax_exog_1', 'sarmax', 'sarmax_exog_1_3', 'sarmax_exog_1_4'],
    'EXPANDING_WINDOW_LSTM': ['lstm', 'bidirectional_lstm', 'lstm_feature_1', 'lstm_feature', 'lstm_features_1_3', 'lstm_features_1_4']
}

# Add model group column to all_models_comparison
def get_model_group(model_name):
    for group, models in model_groups.items():
        # Check if any model from the group is contained in the model_name
        # This handles cases like lstm_ts42_ensemble by checking just for "lstm"
        if any(model in model_name.lower() for model in models):
            return group
    return "OTHER"

# Add this after all_models_comparison is populated
if 'all_models_comparison' in locals() and not all_models_comparison.empty:
    # Add MODEL_GROUP column
    all_models_comparison['MODEL_GROUP'] = all_models_comparison['MODEL'].apply(get_model_group)
    
    # Create filtered dataframes by model group
    model_group_dfs = {}
    for group in model_groups.keys():
        model_group_dfs[group] = all_models_comparison[all_models_comparison['MODEL_GROUP'] == group]
        
    # Print summary statistics by model group
    print(f"Total models analyzed: {len(all_models_comparison)}")
    for group, df in model_group_dfs.items():
        print(f"\n{group}: {len(df)} models")

    # Calculate summary statistics by model group
    model_group_stats = {}
    for group, df in model_group_dfs.items():
        df.loc[:, 'MODEL_NAME'] = df['MODEL'].replace(new_names)
        # Save individual model group dataframes
        output_path = output_folder / f"{group.lower()}_models_comparison.csv"
        df.to_csv(output_path, index=False)
        print(f"\nSaved to {output_path}")

In [ ]:
# Calculate MAE_vs_BEST_STATISTICAL column
if 'all_models_comparison' in locals() and not all_models_comparison.empty:
    # Define statistical models (both rolling window and expanding window)
    statistical_models = (
        model_groups['ROLLING_WINDOW_STATISTICAL'] + 
        model_groups['EXPANDING_WINDOW_STATISTICAL']
    )
    
    # Filter for statistical models only
    statistical_models_df = all_models_comparison[
        all_models_comparison['MODEL_GROUP'].isin(['ROLLING_WINDOW_STATISTICAL', 'EXPANDING_WINDOW_STATISTICAL'])
    ].copy()
    
    # Initialize the new column
    all_models_comparison['MAE_vs_BEST_STATISTICAL'] = np.nan
    
    # For each combination of PRODUCT, PRODUCT_SIGN, and PRICE_TYPE
    for product in products:
        for product_sign in product_signs:
            for price_type in price_types:
                # Filter statistical models for this specific combination
                subset_statistical = statistical_models_df[
                    (statistical_models_df['PRODUCT'] == product) &
                    (statistical_models_df['PRODUCT_SIGN'] == product_sign) &
                    (statistical_models_df['PRICE_TYPE'] == price_type)
                ]
                
                if not subset_statistical.empty:
                    # Find the best statistical model based on combined score
                    best_statistical_model = subset_statistical.loc[
                        subset_statistical['COMBINED_SCORE'].idxmax()
                    ]
                    best_statistical_mae = best_statistical_model['MED_MAE']
                    
                    # Filter all models for this specific combination
                    mask = (
                        (all_models_comparison['PRODUCT'] == product) &
                        (all_models_comparison['PRODUCT_SIGN'] == product_sign) &
                        (all_models_comparison['PRICE_TYPE'] == price_type)
                    )
                    
                    # Calculate percentage difference for all models in this subset
                    subset_indices = all_models_comparison[mask].index
                    for idx in subset_indices:
                        current_mae = all_models_comparison.loc[idx, 'MED_MAE']
                        if best_statistical_mae != 0:  # Avoid division by zero
                            percentage_diff = ((current_mae - best_statistical_mae) / best_statistical_mae) * 100
                            all_models_comparison.loc[idx, 'MAE_vs_BEST_STATISTICAL'] = percentage_diff
                        else:
                            all_models_comparison.loc[idx, 'MAE_vs_BEST_STATISTICAL'] = 0
    
    print("Added MAE_vs_BEST_STATISTICAL column successfully.")
    print(f"Column statistics: Min={all_models_comparison['MAE_vs_BEST_STATISTICAL'].min():.2f}%, Max={all_models_comparison['MAE_vs_BEST_STATISTICAL'].max():.2f}%")

In [ ]:
# Get best models by combined score of errors
best_errors_combined = all_models_comparison.sort_values('COMBINED_SCORE', ascending=False).groupby(
    ['PRODUCT', 'PRODUCT_SIGN', 'PRICE_TYPE']).first()
    
# Get best models by combined score of errors
best_revenue = all_models_comparison.sort_values('REVENUE', ascending=False).groupby(
    ['PRODUCT', 'PRODUCT_SIGN', 'PRICE_TYPE']).first()

best_revenue['MODEL'] = best_revenue['MODEL'].replace(new_names)
best_errors_combined['MODEL'] = best_errors_combined['MODEL'].replace(new_names)

In [ ]:
# Create best models by category (Statistical vs Deep Learning)
if 'all_models_comparison' in locals() and not all_models_comparison.empty:
    
    # Filter statistical models (both rolling window and expanding window)
    statistical_models_df = all_models_comparison[
        all_models_comparison['MODEL_GROUP'].isin(['ROLLING_WINDOW_STATISTICAL', 'EXPANDING_WINDOW_STATISTICAL'])
    ].copy()
    
    # Filter deep learning models
    deep_learning_models_df = all_models_comparison[
        all_models_comparison['MODEL_GROUP'] == 'EXPANDING_WINDOW_LSTM'
    ].copy()
    
    # Statistical models - best by combined score
    if not statistical_models_df.empty:
        best_statistical_combined = statistical_models_df.sort_values('COMBINED_SCORE', ascending=False).groupby(
            ['PRODUCT', 'PRODUCT_SIGN', 'PRICE_TYPE']).first()
        best_statistical_combined['MODEL'] = best_statistical_combined['MODEL'].replace(new_names)
    
    # Statistical models - best by revenue
    if not statistical_models_df.empty:
        best_statistical_revenue = statistical_models_df.sort_values('REVENUE', ascending=False).groupby(
            ['PRODUCT', 'PRODUCT_SIGN', 'PRICE_TYPE']).first()
        best_statistical_revenue['MODEL'] = best_statistical_revenue['MODEL'].replace(new_names)
    
    # Deep learning models - best by combined score
    if not deep_learning_models_df.empty:
        best_deep_learning_combined = deep_learning_models_df.sort_values('COMBINED_SCORE', ascending=False).groupby(
            ['PRODUCT', 'PRODUCT_SIGN', 'PRICE_TYPE']).first()
        best_deep_learning_combined['MODEL'] = best_deep_learning_combined['MODEL'].replace(new_names)
    
    # Deep learning models - best by revenue
    if not deep_learning_models_df.empty:
        best_deep_learning_revenue = deep_learning_models_df.sort_values('REVENUE', ascending=False).groupby(
            ['PRODUCT', 'PRODUCT_SIGN', 'PRICE_TYPE']).first()
        best_deep_learning_revenue['MODEL'] = best_deep_learning_revenue['MODEL'].replace(new_names)
    
    print("Created category-specific best model dataframes.")

In [ ]:
# Replace the existing export section with this expanded version

# Export all results
all_models_comparison.to_csv(output_folder / 'all_models_comparison.csv', index=False)
best_revenue.to_csv(output_folder / 'best_models_by_revenue.csv')
best_errors_combined.to_csv(output_folder / 'best_models_by_combined_score_of_errors.csv')

# Export category-specific best models
if 'best_statistical_combined' in locals():
    best_statistical_combined.to_csv(output_folder / 'best_statistical_models_by_combined_score.csv')
    print(f"Saved best statistical models by combined score to: {output_folder / 'best_statistical_models_by_combined_score.csv'}")

if 'best_statistical_revenue' in locals():
    best_statistical_revenue.to_csv(output_folder / 'best_statistical_models_by_revenue.csv')
    print(f"Saved best statistical models by revenue to: {output_folder / 'best_statistical_models_by_revenue.csv'}")

if 'best_deep_learning_combined' in locals():
    best_deep_learning_combined.to_csv(output_folder / 'best_deep_learning_models_by_combined_score.csv')
    print(f"Saved best deep learning models by combined score to: {output_folder / 'best_deep_learning_models_by_combined_score.csv'}")

if 'best_deep_learning_revenue' in locals():
    best_deep_learning_revenue.to_csv(output_folder / 'best_deep_learning_models_by_revenue.csv')
    print(f"Saved best deep learning models by revenue to: {output_folder / 'best_deep_learning_models_by_revenue.csv'}")

print("All exports completed successfully.")